<a href="https://colab.research.google.com/github/aquastrain/chatbot/blob/main/arxiv_chatbot_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ArXiv CS Chatbot
**NLP Task 4** - Expert chatbot for Computer Science research papers

### Features
- Semantic paper search using TF-IDF
- Abstractive summarisation with BART (facebook/bart-large-cnn)
- Concept explanation and follow-up Q&A with Flan-T5 (google/flan-t5-base)
- Word cloud and keyword visualisation
- Streamlit UI accessible via pyngrok tunnel

**Run all cells top-to-bottom. The last cell prints a public URL.**

## Step 1 - Install dependencies

In [ ]:
%%capture
!pip install streamlit pyngrok transformers torch scikit-learn wordcloud matplotlib pandas

## Step 2 - (Optional) Prepare the arXiv dataset

Skip this step to run with the built-in demo papers.

To use the full dataset:
1. Download `arxiv-metadata-oai-snapshot.json` from https://www.kaggle.com/datasets/Cornell-University/arxiv
2. Upload it to Colab via the Files panel on the left
3. Run the cell below to extract CS papers into a smaller JSONL file

In [ ]:
import json, os

SOURCE = 'arxiv-metadata-oai-snapshot.json'
OUTPUT = 'arxiv_cs_sample.jsonl'
MAX    = 5000

if os.path.exists(SOURCE):
    count = 0
    with open(SOURCE, 'r') as fin, open(OUTPUT, 'w') as fout:
        for line in fin:
            try:
                obj = json.loads(line)
                if obj.get('categories', '').startswith('cs'):
                    fout.write(line)
                    count += 1
                    if count >= MAX:
                        break
            except Exception:
                pass
    print(f'Extracted {count} CS papers -> {OUTPUT}')
else:
    print(f'{SOURCE} not found. Demo mode will be used (built-in papers).')

arxiv-metadata-oai-snapshot.json not found. Demo mode will be used (built-in papers).


## Step 3 - Write the Streamlit app to disk

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import json, re, os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

st.set_page_config(page_title='ArXiv CS Chatbot', page_icon='robot', layout='wide')

DEMO_PAPERS = [
    {'id':'2303.08774','title':'GPT-4 Technical Report',
     'abstract':'We report the development of GPT-4, a large-scale multimodal model which can accept image and text inputs and produce text outputs. GPT-4 exhibits human-level performance on various professional and academic benchmarks.',
     'authors':'OpenAI','categories':'cs.CL','update_date':'2023-03-15'},
    {'id':'1706.03762','title':'Attention Is All You Need',
     'abstract':'The dominant sequence transduction models are based on complex recurrent or convolutional neural networks. We propose the Transformer, a model architecture based solely on attention mechanisms, dispensing with recurrence and convolutions entirely.',
     'authors':'Vaswani et al.','categories':'cs.CL','update_date':'2017-12-06'},
    {'id':'2005.14165','title':'Language Models are Few-Shot Learners',
     'abstract':'We train GPT-3, an autoregressive language model with 175 billion parameters, and test its performance in the few-shot setting on NLP tasks without gradient updates or fine-tuning.',
     'authors':'Brown et al.','categories':'cs.CL','update_date':'2020-07-22'},
    {'id':'1512.03385','title':'Deep Residual Learning for Image Recognition',
     'abstract':'We present a residual learning framework to ease the training of networks that are substantially deeper than those used previously using shortcut connections.',
     'authors':'He et al.','categories':'cs.CV','update_date':'2015-12-10'},
    {'id':'1301.3666','title':'Efficient Estimation of Word Representations in Vector Space',
     'abstract':'We propose two novel model architectures for computing continuous vector representations of words from very large data sets evaluated on a word similarity task.',
     'authors':'Mikolov et al.','categories':'cs.CL','update_date':'2013-09-07'},
    {'id':'1810.04805','title':'BERT: Pre-training of Deep Bidirectional Transformers',
     'abstract':'We introduce BERT which pre-trains deep bidirectional representations from unlabeled text by jointly conditioning on both left and right context in all layers.',
     'authors':'Devlin et al.','categories':'cs.CL','update_date':'2018-10-11'},
    {'id':'2010.11929','title':'An Image is Worth 16x16 Words: Transformers for Image Recognition',
     'abstract':'We apply a pure transformer directly to sequences of image patches for image classification. When pre-trained on large amounts of data it attains excellent results.',
     'authors':'Dosovitskiy et al.','categories':'cs.CV','update_date':'2020-10-22'},
]

@st.cache_resource(show_spinner='Loading summariser (BART)...')
def load_summariser():
    from transformers import pipeline
    return pipeline('summarization', model='facebook/bart-large-cnn', device=-1, truncation=True)

@st.cache_resource(show_spinner='Loading LLM (Flan-T5)...')
def load_llm():
    from transformers import pipeline
    return pipeline('text2text-generation', model='google/flan-t5-base', device=-1, max_new_tokens=300)

@st.cache_data(show_spinner='Loading papers...')
def load_data(path, n):
    records = []
    try:
        with open(path, 'r', encoding='utf-8') as f:
            for line in f:
                obj = json.loads(line)
                if not obj.get('categories','').startswith('cs'):
                    continue
                records.append({
                    'id':          obj.get('id',''),
                    'title':       obj.get('title','').replace('\n',' ').strip(),
                    'abstract':    obj.get('abstract','').replace('\n',' ').strip(),
                    'authors':     obj.get('authors',''),
                    'categories':  obj.get('categories',''),
                    'update_date': obj.get('update_date',''),
                })
                if len(records) >= n:
                    break
    except FileNotFoundError:
        pass
    if len(records) < 5:
        st.warning('arxiv_cs_sample.jsonl not found -- using demo papers.')
        records = DEMO_PAPERS
    return pd.DataFrame(records)

@st.cache_resource
def build_index(_df):
    corpus = (_df['title'] + ' ' + _df['abstract']).tolist()
    vec = TfidfVectorizer(stop_words='english', max_features=20000, ngram_range=(1,2))
    mat = vec.fit_transform(corpus)
    return vec, mat

def search_papers(q, df, vec, mat, k=5):
    scores = cosine_similarity(vec.transform([q]), mat).flatten()
    idx    = np.argsort(scores)[::-1][:k]
    res    = df.iloc[idx].copy()
    res['score'] = scores[idx]
    return res[res['score'] > 0]

def summarise(text, sumr):
    return sumr(text[:1024], max_length=130, min_length=40, do_sample=False)[0]['summary_text']

def explain_concept(concept, ctx, llm):
    prompt = (f'You are an expert computer science researcher. '
              f'Explain the concept "{concept}" based on the following abstract:\n\n'
              f'{ctx[:800]}\n\nGive a clear graduate-level explanation.')
    return llm(prompt)[0]['generated_text']

def answer_question(q, ctx, llm):
    prompt = (f'You are an expert computer science researcher. '
              f'Answer this question based on the context.\n\n'
              f'Context: {ctx[:800]}\n\nQuestion: {q}\n\nAnswer:')
    return llm(prompt)[0]['generated_text']

def show_wordcloud(text, title=''):
    from wordcloud import WordCloud
    import matplotlib.pyplot as plt
    wc  = WordCloud(width=700, height=300, background_color='white',
                    colormap='viridis', max_words=80).generate(text)
    fig, ax = plt.subplots(figsize=(9, 3))
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(title, fontsize=13)
    st.pyplot(fig)
    plt.close(fig)

# ===================== UI =====================
st.title('ArXiv CS Chatbot')
st.caption('Expert chatbot for Computer Science research papers')

with st.sidebar:
    st.header('Settings')
    data_path = st.text_input('arXiv JSONL path', 'arxiv_cs_sample.jsonl')
    n_papers  = st.slider('Max papers to index', 500, 5000, 2000, step=500)
    top_k     = st.slider('Search results to show', 1, 10, 5)
    use_ai    = st.checkbox('Enable AI summarisation & explanation', True)
    st.markdown('---')
    st.markdown(
        '**How to get the full dataset**\n'
        '1. Download from Kaggle (Cornell-University/arxiv)\n'
        '2. Upload the JSON file to Colab\n'
        '3. Run Step 2 cell in the notebook'
    )

df  = load_data(data_path, n_papers)
vec, mat = build_index(df)

if 'history' not in st.session_state: st.session_state.history = []
if 'paper'   not in st.session_state: st.session_state.paper   = None

t1, t2, t3 = st.tabs(['Paper Search', 'Chatbot', 'Visualisation'])

# ---------- Tab 1: Search ----------
with t1:
    st.subheader('Search arXiv CS Papers')
    q = st.text_input('Enter keywords, concepts, or a question:', key='srch')
    if q:
        res = search_papers(q, df, vec, mat, top_k)
        if res.empty:
            st.info('No matching papers found. Try different keywords.')
        else:
            st.success(f'Found {len(res)} paper(s).')
            for _, row in res.iterrows():
                with st.expander(f"{row['title']}  --  {row['categories']}"):
                    st.markdown(f"**Authors:** {row['authors']}")
                    st.markdown(f"**Date:** {row['update_date']}")
                    st.markdown(f"**Abstract:** {row['abstract']}")
                    if use_ai:
                        c1, c2 = st.columns(2)
                        with c1:
                            if st.button('Summarise', key=f"sum_{row['id']}"):
                                with st.spinner('Summarising...'):
                                    sm = summarise(row['abstract'], load_summariser())
                                st.info(sm)
                        with c2:
                            if st.button('Load into Chatbot', key=f"load_{row['id']}"):
                                st.session_state.paper = row.to_dict()
                                st.success('Paper loaded!')

# ---------- Tab 2: Chatbot ----------
with t2:
    st.subheader('Chat with the ArXiv Expert')
    p   = st.session_state.paper
    ctx = (p['abstract'] if p else
           'Computer science covers algorithms, AI, machine learning, '
           'computer networks, databases, and software engineering.')
    if p:
        st.success(f"Active paper: **{p['title']}**")
    else:
        st.info('No paper selected. General CS mode. Load a paper from the Search tab.')

    um = st.chat_input('Ask a question or type a concept to explain...')
    if um:
        if not use_ai:
            reply = 'AI models are disabled. Enable them in the sidebar.'
        else:
            llm = load_llm()
            with st.spinner('Thinking...'):
                if re.match(r'^(what is|explain|define|describe|tell me about)\b', um.lower()):
                    concept = re.sub(r'^(what is|explain|define|describe|tell me about)\s*',
                                     '', um, flags=re.IGNORECASE).strip('?. ')
                    reply = explain_concept(concept, ctx, llm)
                else:
                    reply = answer_question(um, ctx, llm)
        st.session_state.history.append(('user',      um))
        st.session_state.history.append(('assistant', reply))

    for role, msg in st.session_state.history:
        with st.chat_message(role):
            st.markdown(msg)

    if st.button('Clear chat'):
        st.session_state.history = []
        st.rerun()

# ---------- Tab 3: Visualisation ----------
with t3:
    st.subheader('Concept Visualisation')
    vc = st.radio('Visualise:', ['Selected paper', 'Search results', 'Dataset overview'],
                  horizontal=True)

    if vc == 'Selected paper':
        if st.session_state.paper:
            p = st.session_state.paper
            show_wordcloud(p['abstract'], p['title'][:60])
            st.markdown('**Top Keywords (TF-IDF)**')
            v2  = TfidfVectorizer(stop_words='english', max_features=200, ngram_range=(1,2))
            m2  = v2.fit_transform([p['abstract']])
            kws = sorted(zip(v2.get_feature_names_out(), m2.toarray()[0]),
                         key=lambda x: -x[1])[:15]
            st.bar_chart(pd.DataFrame(kws, columns=['Keyword','Score']).set_index('Keyword'))
        else:
            st.info('Select a paper in the Search tab first.')

    elif vc == 'Search results':
        q3 = st.text_input('Query for visualisation:', key='vq')
        if q3:
            r3 = search_papers(q3, df, vec, mat, 10)
            if not r3.empty:
                show_wordcloud(' '.join(r3['abstract'].tolist()), f"Top results for '{q3}'")
                sc = r3[['title','score']].copy()
                sc['title'] = sc['title'].str[:45]
                st.bar_chart(sc.set_index('title'))

    else:
        st.markdown(f'**Total papers indexed:** {len(df)}')
        st.markdown('**Top CS sub-categories**')
        st.bar_chart(df['categories'].str.split().explode().value_counts().head(15))
        show_wordcloud(' '.join(df['abstract'].sample(min(200,len(df))).tolist()),
                       'Dataset Word Cloud')


Writing app.py


## Step 4 - Launch the app via pyngrok

Run this cell. It will print a public URL you can open in any browser.

> **Note:** The first run downloads BART and Flan-T5 models (~1.6 GB total).
> This takes a few minutes on Colab. Subsequent runs reuse the cache.

In [ ]:
import subprocess, time
from pyngrok import ngrok

# Kill any existing Streamlit process
subprocess.run(['pkill', '-f', 'streamlit'], capture_output=True)
time.sleep(2)

# Launch Streamlit in the background
proc = subprocess.Popen(
    ['streamlit', 'run', 'app.py',
     '--server.port', '8501',
     '--server.headless', 'true',
     '--server.enableCORS', 'false',
     '--server.enableXsrfProtection', 'false'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(5)

# !!! IMPORTANT: Replace 'YOUR_AUTH_TOKEN' with your actual ngrok authtoken. !!!
# You can get one by signing up at https://dashboard.ngrok.com/signup
# and then going to https://dashboard.ngrok.com/get-started/your-authtoken
ngrok.set_auth_token("3FJcUNgmZg9rSPQ4PmRfiruk9WP_71w9S6RvGGGuk765JmVia")

# Create a public ngrok tunnel
public_url = ngrok.connect(8501)
print('=' * 60)
print(f'  Chatbot is LIVE at: {public_url}')
print('=' * 60)
print('  Keep this cell running and open the link above in your browser.')

  Chatbot is LIVE at: NgrokTunnel: "https://bruising-rift-supernova.ngrok-free.dev" -> "http://localhost:8501"
  Keep this cell running and open the link above in your browser.
